# TP Breast Cancer — Random Forest, SVM et optimisation PSO
La cible est codée : **0 = maligne**, **1 = bénigne**. La fuite de cible a été corrigée.
Le PSO optimise les hyperparamètres d’un SVM RBF sur les données d’apprentissage uniquement.
Les anciennes métriques binaires et courbes ROC prennent la classe 1 (bénigne) comme positive.
La recherche GridSearch et le PSO maximisent le **F1 de la classe maligne (0)**.
Le test est réservé à la comparaison finale ; ne pas régler le PSO en fonction de ses scores.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split 
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC
from sklearn.metrics import mean_squared_error as MSE
from sklearn.metrics import r2_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import roc_auc_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import ConfusionMatrixDisplay, make_scorer
from sklearn.pipeline import Pipeline


In [ ]:


data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target  # 0 = Maligne, 1 = Bénigne


In [ ]:
df.shape
print(df.head(3))
print(df.tail(3))
df.info()

In [ ]:
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = df.select_dtypes(include=['object', 'category']).columns
print("Variables numériques")
print(numerical_cols)
print("Variables catégorielles")
print(categorical_cols)



In [ ]:

# Pas de colonne 'id' dans le dataset Breast Cancer
# df = df.drop(['id'], axis=1)  # non nécessaire ici


In [ ]:
df.describe()

In [ ]:

print(df['target'].unique())  


In [ ]:
df[df.duplicated()].sum()

In [ ]:
df.isnull().values.sum()

In [ ]:

sns.countplot(data=df, x='target')
plt.xticks([0, 1], ['Maligne (0)', 'Bénigne (1)'])
plt.show();


In [ ]:

cols = df.columns[:3]

# Boucle simple
for col in cols:
    sns.boxplot(x=df['target'], y=df[col])
    plt.title(col)
    plt.show()


In [ ]:
# Plotting a correlation matrix
correlation_matrix = df.corr()

# Display the correlation matrix as a heatmap
plt.figure(figsize=(16, 12))
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', fmt=".2f", cbar=True)
plt.title("Feature Correlation Matrix")
plt.show();

In [ ]:
X = df.drop(columns=['target'])
y = df['target']
assert 'target' not in X.columns
print('X :', X.shape, 'y :', y.shape)


In [ ]:

seed = 42
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2,
                           
                                                random_state = seed, stratify=y)
print('Taille train :', X_train.shape[0])
print('Taille test  :', X_test.shape[0])

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [ ]:

rf = RandomForestClassifier(random_state = seed)


rf.fit(X_train, y_train)


y_pred = rf.predict(X_test)

In [ ]:
cv_rf = cross_val_score(rf, X_train, y_train, cv=5, scoring='accuracy')
print('Scores CV apprentissage :', cv_rf)
print('Moyenne :', cv_rf.mean())


In [ ]:

accuracy_rf = accuracy_score(y_test, y_pred)
print(f"Random Forest Classifier Accuracy: {accuracy_rf:.4f}")
importances_rf = pd.Series(rf.feature_importances_, index = X.columns)
sorted_importances_rf = importances_rf.sort_values()
sorted_importances_rf.plot(kind='barh', color='lightgreen'); plt.show()

In [ ]:

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))

In [ ]:
# Getting the predicted probabilities for the positive class
y_probs = rf.predict_proba(X_test)[:, 1]

# Computing ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_probs)

# Calculating the AUC (Area Under the Curve)
roc_auc = auc(fpr, tpr)

# Plotting ROC curve
plt.figure(figsize=(10, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label='ROC curve (AUC = {:.2f})'.format(roc_auc))
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.show();

In [ ]:

# Prédiction sur l'ensemble de test
y_pred = rf.predict(X_test)

# Calcul de la matrice de confusion
cm = confusion_matrix(y_test, y_pred)

# Affichage de la matrice de confusion
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Maligne', 'Bénigne'],
            yticklabels=['Maligne', 'Bénigne'])
plt.title('Confusion Matrix')
plt.xlabel('Prédit')
plt.ylabel('Réel')
plt.show()


In [ ]:
# Training accuracy
train_accuracy = accuracy_score(y_train, rf.predict(X_train))

# Testing accuracy
test_accuracy = accuracy_score(y_test, y_pred)

# Plotting Train Accuracy vs Test Accuracy
plt.figure(figsize=(6, 4))
accuracy_values = [train_accuracy, test_accuracy]
accuracy_labels = ['Train Accuracy', 'Test Accuracy']

plt.bar(accuracy_labels, accuracy_values, color=['blue', 'orange'])
plt.ylabel('Accuracy')
plt.title('Train Accuracy vs Test Accuracy')
plt.ylim(0, 1)
plt.show();

In [ ]:
# La standardisation est réapprise dans chaque pli de validation.
svm_pipeline = Pipeline([('scaler', StandardScaler()),
                         ('svm', SVC(probability=True, random_state=seed))])
para_grid = {'svm__C': [0.1, 1, 10], 'svm__kernel': ['linear', 'rbf']}
scorer_maligne = make_scorer(f1_score, pos_label=0)
cv_search = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
grid = GridSearchCV(svm_pipeline, para_grid, cv=cv_search,
                    scoring=scorer_maligne, n_jobs=1)
grid.fit(X_train, y_train)
print('Meilleurs paramètres SVM :', grid.best_params_)
best_svm = grid.best_estimator_
y_pred_svm = best_svm.predict(X_test)


In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred_svm))
print("Precision:", precision_score(y_test, y_pred_svm))
print("Recall:", recall_score(y_test, y_pred_svm))
print("F1-score:", f1_score(y_test, y_pred_svm))

In [ ]:
cm = confusion_matrix(y_test, y_pred_svm)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title("Matrice de confusion - SVM")
plt.show()


In [ ]:
y_probs_svm = best_svm.predict_proba(X_test)[:, 1]

# Calcul ROC
fpr, tpr, thresholds = roc_curve(y_test, y_probs_svm)

# AUC
roc_auc = roc_auc_score(y_test, y_probs_svm)

# Plot
plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, label=f"SVM (AUC = {roc_auc:.3f})")
plt.plot([0,1], [0,1], linestyle='--')  # ligne aléatoire
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - SVM")
plt.legend()
plt.grid()
plt.show()

In [ ]:
y_probs_rf = rf.predict_proba(X_test)[:, 1]
y_probs_svm = best_svm.predict_proba(X_test)[:, 1]

# ROC RF
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_probs_rf)
auc_rf = roc_auc_score(y_test, y_probs_rf)

# ROC SVM
fpr_svm, tpr_svm, _ = roc_curve(y_test, y_probs_svm)
auc_svm = roc_auc_score(y_test, y_probs_svm)

# Plot
plt.figure(figsize=(8,6))

plt.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC = {auc_rf:.3f})")
plt.plot(fpr_svm, tpr_svm, label=f"SVM (AUC = {auc_svm:.3f})")

# ligne aléatoire
plt.plot([0,1], [0,1], linestyle='--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Comparaison ROC : SVM vs Random Forest")
plt.legend()
plt.grid()

plt.show()

## PSO : optimisation par essaim particulaire
Chaque particule est une solution : **[log10(C), log10(gamma)]**.
- `C` règle la pénalisation des erreurs du SVM : recherche de 0.01 à 1000.
- `gamma` règle la portée du noyau RBF : recherche de 0.0001 à 10.
- Fitness = moyenne du F1 maligne sur 5 plis fixes de validation croisée, dans le train.
- Chaque particule mémorise sa meilleure position (`pbest`) et suit la meilleure de l’essaim (`gbest`).

Mise à jour :
$$v \leftarrow wv+c_1r_1(pbest-x)+c_2r_2(gbest-x),\qquad x\leftarrow x+v.$$
`w` conserve une partie du mouvement ; `c1` représente l’expérience individuelle ; `c2` le partage collectif.
Les positions sont bornées. Le meilleur résultat trouvé n’est pas une garantie d’optimum global.
Implémentation NumPy sans bibliothèque PSO supplémentaire.


In [ ]:
# Évaluation d'une position, sans consulter X_test ou y_test.
cv_pso = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
scorer_pso = make_scorer(f1_score, pos_label=0)
cache_pso = {}

def fitness_pso(position):
    key = tuple(float(v) for v in position)
    if key not in cache_pso:
        C, gamma = 10.0 ** np.asarray(position)
        model = Pipeline([
            ('scaler', StandardScaler()),
            ('svm', SVC(kernel='rbf', C=C, gamma=gamma))
        ])
        scores = cross_val_score(model, X_train, y_train,
                                 cv=cv_pso, scoring=scorer_pso, n_jobs=1)
        cache_pso[key] = float(scores.mean())
    return cache_pso[key]


In [ ]:
# 12 particules, 15 déplacements : au plus 192 évaluations, soit 960 fits CV.
rng = np.random.default_rng(seed)
n_particles, n_iterations = 12, 15
lower = np.array([-2.0, -4.0])
upper = np.array([3.0, 1.0])
w, c1, c2 = 0.7, 1.5, 1.5
vmax = 0.2 * (upper - lower)
positions = rng.uniform(lower, upper, size=(n_particles, 2))
velocities = rng.uniform(-vmax, vmax, size=(n_particles, 2))
pbest = positions.copy()
pbest_scores = np.array([fitness_pso(p) for p in positions])
best_index = np.argmax(pbest_scores)
gbest = pbest[best_index].copy()
gbest_score = float(pbest_scores[best_index])
history = [gbest_score]

for iteration in range(n_iterations):
    r1 = rng.random((n_particles, 2))
    r2 = rng.random((n_particles, 2))
    velocities = (w * velocities + c1 * r1 * (pbest - positions)
                  + c2 * r2 * (gbest - positions))
    velocities = np.clip(velocities, -vmax, vmax)
    positions = np.clip(positions + velocities, lower, upper)
    scores = np.array([fitness_pso(p) for p in positions])
    improved = scores > pbest_scores
    pbest[improved] = positions[improved]
    pbest_scores[improved] = scores[improved]
    best_index = np.argmax(pbest_scores)
    if pbest_scores[best_index] > gbest_score:
        gbest = pbest[best_index].copy()
        gbest_score = float(pbest_scores[best_index])
    history.append(gbest_score)
    print(f'Itération {iteration+1:02d} : meilleur F1 CV maligne = {gbest_score:.4f}')

best_C, best_gamma = 10.0 ** gbest
print(f'C = {best_C:.6g}, gamma = {best_gamma:.6g}')
print('Positions évaluées :', len(cache_pso))
assert np.all(gbest >= lower) and np.all(gbest <= upper)
assert np.all(np.diff(history) >= -1e-12)


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(len(history)), history, marker='o')
plt.xlabel('Itération (0 = initialisation)')
plt.ylabel('Meilleur F1 maligne en validation croisée')
plt.title('Convergence du PSO')
plt.grid(alpha=0.3)
plt.show()


In [ ]:
# Réapprendre sur tout le train avec les paramètres retenus.
pso_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', C=best_C, gamma=best_gamma,
                probability=True, random_state=seed))
])
pso_svm.fit(X_train, y_train)
y_pred_pso = pso_svm.predict(X_test)

rows = []
for name, pred in [('Random Forest', y_pred),
                   ('SVM GridSearch', y_pred_svm), ('SVM PSO', y_pred_pso)]:
    rows.append({'Modèle': name, 'Accuracy test': accuracy_score(y_test, pred),
                 'Précision maligne': precision_score(y_test, pred, pos_label=0, zero_division=0),
                 'Rappel maligne': recall_score(y_test, pred, pos_label=0, zero_division=0),
                 'F1 maligne': f1_score(y_test, pred, pos_label=0, zero_division=0)})
comparison_pso = pd.DataFrame(rows)
print(comparison_pso.round(4).to_string(index=False))
print('\nRapport SVM PSO :')
print(classification_report(y_test, y_pred_pso,
      target_names=['Maligne', 'Bénigne'], zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_pso,
                       display_labels=['Maligne', 'Bénigne'], cmap='Blues')
plt.title('SVM optimisé par PSO — test')
plt.show()


### Interprétation
La convergence montre le meilleur score **de validation**, pas celui du test.
Le tableau compare les modèles sur le même test. PSO peut améliorer le SVM, sans garantie.
GridSearch et PSO explorent des espaces et des budgets différents : cette comparaison ne prouve pas qu’une méthode d’optimisation est supérieure.
Pour une étude de robustesse, répéter l’essaim avec plusieurs graines en gardant le test réservé et rapporter la dispersion des scores de validation.
